# Module 12: Geographic Visualization with Matplotlib & Cartopy
## Publication-Quality Rainfall Maps of Ethiopia

This module focuses on geographic visualization: map projections, cartographic elements, regional zooming, multi-panel maps, and overlay techniques using CHIRPS rainfall data with cartopy.

## Setup

Load required libraries and the prepared datasets.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import FancyBboxPatch, Polygon
from matplotlib.path import Path
from scipy import ndimage
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

In [ ]:
DATA_DIR = '../datasets'
CHIRPS_PATH = '../chirps_1981_2022.nc'

ds = xr.open_dataset(CHIRPS_PATH, engine='netcdf4')
eth = xr.open_dataset(f'{DATA_DIR}/chirps_ethiopia.nc', engine='netcdf4')
clim = xr.open_dataset(f'{DATA_DIR}/chirps_climatology.nc', engine='netcdf4')
annual = xr.open_dataset(f'{DATA_DIR}/chirps_annual.nc', engine='netcdf4')

mean_annual = annual.precip.mean(dim='time')
eth_mean = eth.precip.mean(dim='time')

print(f'Full: {ds.precip.shape}')
print(f'Ethiopia subset: {eth.precip.shape}')
print(f'Mean annual shape: {mean_annual.shape}')

## 1. Cartopy Map Projections

Cartopy provides a wide range of map projections. We'll compare three common projections: PlateCarree, Mercator, and Orthographic.

In [ ]:
projections = {
    'PlateCarree': ccrs.PlateCarree(central_longitude=40),
    'Mercator': ccrs.Mercator(central_longitude=40),
    'Orthographic': ccrs.Orthographic(central_longitude=40, central_latitude=8)
}

fig, axes = plt.subplots(1, 3, figsize=(18, 8),
                         subplot_kw={'projection': ccrs.PlateCarree()})

for idx, (proj_name, proj) in enumerate(projections.items()):
    ax = axes[idx]
    ax.projection = proj

    data_proj = ccrs.PlateCarree()
    ax.pcolormesh(eth.longitude, eth.latitude, eth_mean.values.T,
                  cmap='viridis', shading='auto',
                  vmin=0, vmax=250, transform=data_proj)

    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.LAKES, alpha=0.3, color='lightblue')
    ax.add_feature(cfeature.OCEAN, alpha=0.1, color='azure')

    if proj_name != 'Orthographic':
        ax.set_extent([30, 55, -5, 20], crs=data_proj)

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False

    ax.set_title(proj_name, fontsize=13, fontweight='bold')

fig.suptitle('Comparison of Map Projections for Rainfall Data',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 2. Cartographic Elements

Professional maps require coastlines, borders, rivers, lakes, gridlines, and colorbars with proper labeling.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10),
                       subplot_kw={'projection': ccrs.PlateCarree(central_longitude=40)})

pcm = ax.pcolormesh(eth.longitude, eth.latitude, eth_mean.values.T,
                    cmap='Spectral_r', shading='auto',
                    vmin=0, vmax=250,
                    transform=ccrs.PlateCarree())

ax.add_feature(cfeature.COASTLINE, linewidth=1.0, edgecolor='#333333')
ax.add_feature(cfeature.BORDERS, linewidth=0.8, edgecolor='#333333', alpha=0.8)
ax.add_feature(cfeature.LAKES, alpha=0.5, color='#aad4e5',
               edgecolor='#333333', linewidth=0.5)
ax.add_feature(cfeature.RIVERS, linewidth=0.5, alpha=0.6, color='#5dade2')
ax.add_feature(cfeature.OCEAN, alpha=0.2, color='#ebf5fb')

ax.set_extent([33, 48, 3, 15], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels=True, linewidth=0.4, alpha=0.5, linestyle='--', color='gray')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 10, 'color': '#333333'}
gl.ylabel_style = {'size': 10, 'color': '#333333'}

cbar = plt.colorbar(pcm, ax=ax, orientation='horizontal',
                    pad=0.06, aspect=40, shrink=0.75,
                    ticks=np.arange(0, 301, 50))
cbar.set_label('Mean Monthly Precipitation (mm/month)', fontsize=12, fontweight='bold')
cbar.ax.tick_params(labelsize=10)

ax.set_title('Mean Monthly Rainfall over Ethiopia (CHIRPS 1981-2022)',
             fontsize=15, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

## 3. Regional Zoom: Ethiopia and the Horn of Africa

Create maps at different spatial scales: the full Horn of Africa, Ethiopia, and a zoomed region.

In [ ]:
extents = [
    ('Horn of Africa', [30, 55, -5, 20]),
    ('Ethiopia', [33, 48, 3, 15]),
    ('Ethiopian Highlands', [35, 42, 6, 14])
]

fig, axes = plt.subplots(1, 3, figsize=(18, 8),
                         subplot_kw={'projection': ccrs.PlateCarree()})

for idx, (name, extent) in enumerate(extents):
    ax = axes[idx]

    ax.pcolormesh(eth.longitude, eth.latitude, eth_mean.values.T,
                  cmap='viridis', shading='auto', vmin=0, vmax=250,
                  transform=ccrs.PlateCarree())

    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.LAKES, alpha=0.3, color='lightblue')

    ax.set_extent(extent, crs=ccrs.PlateCarree())

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False

    ax.set_title(name, fontsize=13, fontweight='bold')

fig.suptitle('Multi-Scale Mapping of Ethiopian Rainfall',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Multiple Subplot Maps: Seasonal Rainfall

A 2×2 grid of maps showing rainfall for each season, with shared color scale.

In [ ]:
season_months = {
    'DJF (Dry)': [12, 1, 2],
    'MAM (Long Rain)': [3, 4, 5],
    'JJA (Main Rain)': [6, 7, 8],
    'SON (Short Rain)': [9, 10, 11]
}

seasonal = {}
for name, months in season_months.items():
    mask = eth.time.dt.month.isin(months)
    seasonal[name] = eth.precip.where(mask).mean(dim='time')

vmin_all = 0
vmax_all = max(s.max().values for s in seasonal.values())

fig, axes = plt.subplots(2, 2, figsize=(16, 14),
                         subplot_kw={'projection': ccrs.PlateCarree()})
axes = axes.flatten()

for idx, (name, data) in enumerate(seasonal.items()):
    ax = axes[idx]

    pcm = ax.pcolormesh(data.longitude, data.latitude, data.values.T,
                        cmap='RdYlBu_r', shading='auto',
                        vmin=vmin_all, vmax=vmax_all,
                        transform=ccrs.PlateCarree())

    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.LAKES, alpha=0.3, color='lightblue')

    ax.set_extent([33, 48, 3, 15], crs=ccrs.PlateCarree())

    gl = ax.gridlines(draw_labels=True, linewidth=0.2, alpha=0.3, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 8}
    gl.ylabel_style = {'size': 8}

    ax.set_title(name, fontsize=12, fontweight='bold', pad=8)

cbar = fig.colorbar(pcm, ax=axes, orientation='horizontal',
                    pad=0.05, aspect=50, shrink=0.6)
cbar.set_label('Precipitation (mm/month)', fontsize=12)

fig.suptitle('Seasonal Rainfall Patterns over Ethiopia (CHIRPS 1981-2022)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Region Annotation Overlay

Overlay Ethiopian administrative region boundaries as rectangles on the rainfall map.

In [ ]:
regions_bbox = {
    'Tigray': {'lon': (36.5, 39.8), 'lat': (12.5, 14.9)},
    'Amhara': {'lon': (36.5, 40.5), 'lat': (9.5, 12.5)},
    'Oromia': {'lon': (34.5, 43.0), 'lat': (3.5, 9.5)},
    'SNNPR': {'lon': (34.5, 39.5), 'lat': (4.5, 8.0)},
    'Somali': {'lon': (41.0, 48.0), 'lat': (3.5, 11.0)},
    'Afar': {'lon': (39.8, 42.5), 'lat': (8.5, 14.5)}
}

fig, ax = plt.subplots(figsize=(12, 10),
                       subplot_kw={'projection': ccrs.PlateCarree()})

ax.pcolormesh(eth.longitude, eth.latitude, eth_mean.values.T,
              cmap='YlGnBu', shading='auto', vmin=0, vmax=250,
              transform=ccrs.PlateCarree())

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.LAKES, alpha=0.3, color='lightblue')
ax.set_extent([33, 48, 3, 15], crs=ccrs.PlateCarree())

colors_reg = plt.cm.Set1(np.linspace(0, 1, len(regions_bbox)))

for idx, (name, bbox) in enumerate(regions_bbox.items()):
    x0, x1 = bbox['lon']
    y0, y1 = bbox['lat']
    rect = plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                         linewidth=2, edgecolor=colors_reg[idx],
                         facecolor='none', linestyle='--',
                         transform=ccrs.PlateCarree())
    ax.add_patch(rect)
    ax.text(x0 + (x1 - x0) / 2, y0 + (y1 - y0) / 2, name,
            transform=ccrs.PlateCarree(), fontsize=9,
            fontweight='bold', color=colors_reg[idx],
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2',
                      facecolor='white', edgecolor='none', alpha=0.7))

gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle='--')
gl.top_labels = False
gl.right_labels = False

ax.set_title('Ethiopian Administrative Regions with Rainfall',
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Multi-Year Comparison Maps

Compare annual rainfall maps for selected years to visualize interannual variability.

In [ ]:
select_years = [1984, 1997, 2000, 2015]

fig, axes = plt.subplots(2, 2, figsize=(16, 12),
                         subplot_kw={'projection': ccrs.PlateCarree()})
axes = axes.flatten()

for idx, year in enumerate(select_years):
    ax = axes[idx]

    annual_year = annual.precip.sel(time=annual.time.dt.year == year).mean(dim='time')
    annual_sub = annual_year.sel(latitude=slice(15, 3), longitude=slice(33, 48))

    lon_yr = annual_sub.longitude.values
    lat_yr = annual_sub.latitude.values

    pcm = ax.pcolormesh(lon_yr, lat_yr, annual_sub.values.T,
                        cmap='YlOrRd', shading='auto',
                        vmin=0, vmax=300,
                        transform=ccrs.PlateCarree())

    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.LAKES, alpha=0.3, color='lightblue')

    ax.set_extent([33, 48, 3, 15], crs=ccrs.PlateCarree())

    gl = ax.gridlines(draw_labels=True, linewidth=0.2, alpha=0.3, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False

    ax.set_title(f'Rainfall {year}', fontsize=13, fontweight='bold')

cbar = fig.colorbar(pcm, ax=axes, orientation='horizontal',
                    pad=0.05, aspect=50, shrink=0.6)
cbar.set_label('Mean Monthly Precipitation (mm/month)', fontsize=11)

fig.suptitle('Annual Rainfall Comparison: Selected Years',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Elevation Profile (Conceptual)

Since we don't have elevation data, we use rainfall as a proxy to create a cross-sectional profile across Ethiopia, demonstrating how to create topographic-like profile plots.

In [ ]:
profile_lat = 9.0  # approximate latitude through Addis Ababa
profile_data = eth_mean.sel(latitude=profile_lat, method='nearest')

fig, (ax_map, ax_profile) = plt.subplots(
    1, 2, figsize=(16, 6),
    gridspec_kw={'width_ratios': [1.5, 1]},
    subplot_kw={'projection': ccrs.PlateCarree()}
)

pcm = ax_map.pcolormesh(eth.longitude, eth.latitude, eth_mean.values.T,
                        cmap='viridis', shading='auto', vmin=0, vmax=250,
                        transform=ccrs.PlateCarree())
ax_map.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax_map.add_feature(cfeature.BORDERS, linewidth=0.5)
ax_map.add_feature(cfeature.LAKES, alpha=0.3, color='lightblue')
ax_map.set_extent([33, 48, 3, 15], crs=ccrs.PlateCarree())

ax_map.axhline(profile_lat, color='red', linestyle='--', linewidth=2,
               transform=ccrs.PlateCarree(), label=f'Profile at {profile_lat}°N')
ax_map.legend(loc='lower right', fontsize=10)

gl = ax_map.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle='--')
gl.top_labels = False
gl.right_labels = False
ax_map.set_title('Map with Profile Line', fontsize=12, fontweight='bold')

ax_profile.fill_between(profile_data.longitude.values,
                        profile_data.values, 0,
                        color='darkgreen', alpha=0.4)
ax_profile.plot(profile_data.longitude.values, profile_data.values,
                'g-', linewidth=2)
ax_profile.set_xlabel('Longitude', fontsize=11)
ax_profile.set_ylabel('Precipitation (mm/month)', fontsize=11)
ax_profile.set_title(f'Rainfall Profile at {profile_lat}°N', fontsize=12, fontweight='bold')
ax_profile.grid(True, alpha=0.3)
ax_profile.set_xlim(profile_data.longitude.min(), profile_data.longitude.max())

plt.tight_layout()
plt.show()

## 8. Publication-Quality Map

A polished, publication-ready map with all cartographic elements, annotations, and professional styling.

In [ ]:
fig = plt.figure(figsize=(16, 12))

ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=40))

pcm = ax.pcolormesh(eth.longitude, eth.latitude, eth_mean.values.T,
                    cmap='Spectral_r', shading='auto',
                    vmin=0, vmax=250,
                    transform=ccrs.PlateCarree())

ax.add_feature(cfeature.COASTLINE, linewidth=1.2, edgecolor='#2c3e50')
ax.add_feature(cfeature.BORDERS, linewidth=0.8, edgecolor='#2c3e50', alpha=0.8)
ax.add_feature(cfeature.LAKES, alpha=0.6, color='#aed6f1',
               edgecolor='#2c3e50', linewidth=0.5)
ax.add_feature(cfeature.RIVERS, linewidth=0.4, alpha=0.5, color='#5dade2')
ax.add_feature(cfeature.OCEAN, alpha=0.3, color='#d6eaf8')

ax.set_extent([33, 48, 3, 15], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.6,
                  linestyle='--', color='#566573')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 11, 'fontweight': 'bold'}
gl.ylabel_style = {'size': 11, 'fontweight': 'bold'}

cax = fig.add_axes([0.25, 0.08, 0.5, 0.02])
cbar = fig.colorbar(pcm, cax=cax, orientation='horizontal',
                    ticks=np.arange(0, 301, 50))
cbar.set_label('Mean Monthly Precipitation (mm/month)',
               fontsize=13, fontweight='bold')
cbar.ax.tick_params(labelsize=11)

ax.set_title('Mean Monthly Rainfall over Ethiopia\nCHIRPS Version 2.0 (1981-2022)',
             fontsize=16, fontweight='bold', pad=15)

text = ax.text(0.01, -0.12, 'Data: CHIRPS v2.0, Climate Hazards Group, UCSB | ',
               'Projection: PlateCarree',
               transform=ax.transAxes, fontsize=9, fontstyle='italic',
               color='#566573')

plt.savefig(f'{DATA_DIR}/../notebooks/ethiopia_rainfall_map.png',
            dpi=300, bbox_inches='tight', facecolor='white')
print('Publication-quality map saved!')
plt.show()

## 9. Shapefile/GeoJSON Overlay (Conceptual)

If administrative boundary shapefiles are available, they can be loaded and overlaid on the rainfall map. Here's the pattern you would use:

```python
import geopandas as gpd
from cartopy.io.shapereader import Reader

# Option 1: Using GeoPandas
# gdf = gpd.read_file('ethiopia_regions.geojson')
# gdf.boundary.plot(ax=ax, edgecolor='black', linewidth=0.8, transform=ccrs.PlateCarree())

# Option 2: Using cartopy's shapereader
# from cartopy.io.shapereader import Reader
# ax.add_geometries(Reader('ethiopia_regions.shp').geometries(),
#                  ccrs.PlateCarree(), edgecolor='black',
#                  facecolor='none', linewidth=1.0)
```

## Exercises

1. **Projection Experiment**: Create the same rainfall map in 5 different projections (PlateCarree, Mercator, Orthographic, LambertAzimuthalEqualArea, Robinson) and compare the visual appearance.
2. **Seasonal Subplots with Contours**: Add contour lines over the seasonal pcolormesh maps.
3. **Anomaly Map**: Create a map showing the difference between El Niño and La Niña composite rainfall patterns.
4. **City Markers**: Add markers for major Ethiopian cities (Addis Ababa, Dire Dawa, Mekelle, Bahir Dar, Jimma) with labels.
5. **Inset Map**: Create a map of Ethiopia with a small inset showing its location within Africa.

### Mini-Project: Publication-Ready Rainfall Atlas

Create a multi-panel figure suitable for a scientific publication including:
- Main panel: Mean annual rainfall map with contour lines, coastlines, borders, lakes, rivers, and gridlines
- Four seasonal panels arranged in a 2×2 grid below the main map
- City markers and labels for at least 5 Ethiopian cities
- A colorbar with custom ticks and label
- A north arrow and scale bar
- Proper title, data citation, and projection information
- Export at 300 dpi as a TIFF or PNG